<a href="https://colab.research.google.com/github/Thrysoe/north_diagnostics/blob/main/create_image_sequence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
"""
Take the data from the .txt files generated by extract_all_data.py (in the data folder).
Plot it into images and then a video. The obtained plots are stored in a sub folder the Figure folder created if not existing.
Can be raw or fluctuating data.
"""

#Get the other programs and modules
!git clone https://github.com/thrysoe/north_diagnostics.git -q
!pip install nptdms -q

#Import all useful libraries
import nptdms
import numpy as np
import matplotlib.pyplot as plt
import scipy.interpolate as sci
import os
import cv2
from north_diagnostics.diagnostics import Probe

# Define all useful functions
def plot_2D_data(data, path_to_figure, time, vmin, vmax, bias_type, data_type):
  """
  Takes the data from the txt files and probe position file to build the vessel, the probes and the movie of raw or
  fluctuating data.
  """
  #Create figure
  fig = plt.figure(figsize=(10,10))

  #Plot the vessel and the probes
  theta = np.linspace(0, 2*np.pi, 250)
  x_loc = 250 + 125*np.cos(theta)
  y_loc = 125*np.sin(theta)
  plt.plot(x_loc, y_loc, label='Vessel boundaries', color='black')

  #Plot the probes and get their positions
  r, z = [], []
  for i in range(Probe.TOTAL_PROBES):
    probe = Probe(path = path_to_data, shot = shot, number = i + 1, caching = True)
    x_p, y_p = probe.position['r'], probe.position['z']
    r.append(x_p)
    z.append(y_p)
    plt.plot(x_p, y_p, marker='o', markeredgecolor='k', markerfacecolor='w' if activated_probes[i] else 'r')
    plt.text(x_p, y_p-10, str(i+1), color='black', fontsize=8)

  #Add all data at this time on the plot
  curr_data = data[time,1:]

  #Creates the grid and the plot area
  grid_r, grid_z = np.meshgrid(np.linspace(250-125, 250+125, 100), np.linspace(-125, 125, 100))
  mask = (grid_r - 250)**2 + (grid_z - 0)**2 > 125**2

  #Computing the map for a contour data plot
  c_interpolator = sci.RBFInterpolator(np.column_stack((r, z)), curr_data, neighbors=7)
  grid_c = c_interpolator(np.column_stack((grid_r.ravel(), grid_z.ravel()))).reshape(grid_r.shape)
  grid_c[mask] = np.nan  # Set outside the circle to NaN

  #Creates the data plot with its colorbar
  contourf = plt.contourf(grid_r, grid_z, grid_c, levels = np.linspace(vmin, vmax, 50), cmap = 'inferno_r')
  plt.colorbar(contourf, orientation='vertical', label=f"{data_type} {bias_type} SI")

  #Add a legend, save and close
  plt.axis('equal')
  plt.xlim((250-140, 250+140))
  plt.xlabel('r (mm)')
  plt.ylabel('z (mm)')
  plt.title(f"Shot {shot} {data_type} {bias_type} at time {data[time,0]} s")
  plt.legend()
  plt.savefig(f"{path_to_figure}/{shot}_{bias_type}_{data_type}/{time}.png")
  plt.close(fig=fig)
  return f"image {str(time)} processed"

def video_2D(data, path_to_figure, bias_type, data_type, fps):
  """
  Takes the images in the appropriate figure folder and concanate them into a video .avi file.
  This function assumes that the images have already been generated and saved by the previous function.
  """
  #Create figure
  image_folder = f"{path_to_figure}/{shot}_{bias_type}_{data_type}/"
  video_name = f"{path_to_figure}/{shot}_{bias_type}_{data_type}/{shot}_{bias_type}_{data_type}.avi"
  images = [img for img in os.listdir(image_folder) if img.endswith((".jpg", ".jpeg", ".png"))]

  # Set frame from the first image
  frame = cv2.imread(os.path.join(image_folder, images[0]))
  height, width, layers = frame.shape

  # Video writer to create .avi file
  video = cv2.VideoWriter(video_name, cv2.VideoWriter_fourcc(*'DIVX'), fps, (width, height))

  # Appending images to video
  for image in images:
    video.write(cv2.imread(os.path.join(image_folder, image)))

  # Release the video file
  video.release()
  cv2.destroyAllWindows()
  return f"Video is generated in the {image_folder} folder"

#Main program: plot all machine and some probe data to verify that the shot "looks fine"
if __name__=="__main__":
  #Input parameters
  shot = 9774
  bias_type = 'temperature' #Probes can be biased to measure 'density' or 'temperature' (the same bias is applied on every probe)
  data_type = 'raw' #We may be interested in the raw data or in their fluctuations. As a consequence, put 'raw' or 'fluctuations'.
  path_to_data = './north_diagnostics/Data/'
  path_to_figure = './north_diagnostics/Figures/'
  fps = 1 #frame rate of the video

  #Extract and cleaning data from .txt file
  data=np.genfromtxt(f"{path_to_data}probe_data{shot}.txt", delimiter=';', skip_header=1)
  if data_type=='fluctuations':
    data = data - np.mean(data, axis=1, keepdims=True)
  print(f"Data shot {shot} ({data_type}) loaded with success")

  #Testing activated probes
  activated_probes = []
  for i in range(Probe.TOTAL_PROBES):
    if np.mean(data[:,i+1]) != 0:
      activated_probes.append(True)
    else:
      activated_probes.append(False)

  #Parameters for plotting a single colorbar
  k = 2
  mean, std = np.mean(data[:,1:]), np.std(data[:,1:])
  vmin, vmax = mean - 1*std, mean + 10*std #To be modified when the temperature data won't be so weird

  #Create a folder to store data if it doesn't exist
  if not os.path.exists(f"{path_to_figure}/{shot}_{bias_type}_{data_type}/"):
    os.makedirs(f"{path_to_figure}/{shot}_{bias_type}_{data_type}/")
    print("A new directory for storing data was created")

  #Plot all images of the vessel + the probes + the data for each time
  for i in range(6): #range(len(data[:,0])):
    output=plot_2D_data(data, path_to_figure, i, vmin, vmax, bias_type, data_type)
    print(output)

  #Save those images in a movie (.avi or .tiff) => to be found in the figure folder
  output = video_2D(data, path_to_figure, bias_type, data_type, fps)
  print(output)

fatal: destination path 'north_diagnostics' already exists and is not an empty directory.
Data shot 9774 (raw) loaded with success
image 0 processed
image 1 processed
image 2 processed
image 3 processed
image 4 processed
image 5 processed
Video is generated in the ./north_diagnostics/Figures//9774_temperature_raw/ folder
